#  <center> Speech Emotion Recognition <center>
### I am going to build a speech emotion detection classifier.
### But first we need to learn about what is speech recognition (SER) and why are we building this project? Well, few of the reasons are-

#### First, lets define SER i.e. Speech Emotion Recognition.
* Speech Emotion Recognition, abbreviated as SER, is the act of attempting to recognize human emotion and affective states from speech. This is capitalizing on the fact that voice often reflects underlying emotion through tone and pitch. This is also the phenomenon that animals like dogs and horses employ to be able to understand human emotion.

#### Why we need it?

1. Emotion recognition is the part of speech recognition which is gaining more popularity and need for it increases enormously. Although there are methods to recognize emotion using machine learning techniques, this project attempts to use deep learning to recognize the emotions from data.

2. SER(Speech Emotion Recognition) is used in call center for classifying calls according to emotions and can be used as the performance parameter for conversational analysis thus identifying the unsatisfied customer, customer satisfaction and so on.. for helping companies improving their services

3. It can also be used in-car board system based on information of the mental state of the driver can be provided to the system to initiate his/her safety preventing accidents to happen

#### Datasets used in this project

* Crowd-sourced Emotional Mutimodal Actors Dataset (Crema-D)
* Ryerson Audio-Visual Database of Emotional Speech and Song (Ravdess)
* Surrey Audio-Visual Expressed Emotion (Savee)
* Toronto emotional speech set (Tess)

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import glob
import time

# librosa is a Python library for analyzing audio and music. It can be used to extract the data from the audio files.
import librosa
import librosa.display
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# to play the audio files
from IPython.display import Audio

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import joblib

import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 

## Data Preparation
* As we are working with datasets, we will create a dataframe storing all emotions of the data with their paths.
* We will use this dataframe to extract features for our model training.

In [ ]:
# Paths for local dataset folders
Ravdess = "data/"
Crema = "crema/"
Tess = "tess/"
Savee = "savee/"

##  <center> 1. Ravdess Dataframe <center>
Here is the filename identifiers as per the official RAVDESS website:

* Modality (01 = full-AV, 02 = video-only, 03 = audio-only).
* Vocal channel (01 = speech, 02 = song).
* Emotion (01 = neutral, 02 = calm, 03 = happy, 04 = sad, 05 = angry, 06 = fearful, 07 = disgust, 08 = surprised).
* Emotional intensity (01 = normal, 02 = strong). NOTE: There is no strong intensity for the 'neutral' emotion.
* Statement (01 = "Kids are talking by the door", 02 = "Dogs are sitting by the door").
* Repetition (01 = 1st repetition, 02 = 2nd repetition).
* Actor (01 to 24. Odd numbered actors are male, even numbered actors are female).

So, here's an example of an audio filename: `03-01-06-01-02-01-12.wav`
This means the meta data for the audio file is:

* Audio-only (03)
* Speech (01)
* Fearful (06)
* Normal intensity (01)
* Statement "dogs" (02)
* 1st Repetition (01)
* 12th Actor (12) - Female (as the actor ID number is even)

In [ ]:
ravdess_directory_list = os.listdir(Ravdess)

file_emotion = []
file_path = []
for dir in ravdess_directory_list:
    if not os.path.isdir(os.path.join(Ravdess, dir)):
        continue
    actor = os.listdir(os.path.join(Ravdess, dir))
    for file in actor:
        part = file.split('.')[0]
        part = part.split('-')
        if len(part) >= 3:
            # third part represents the emotion associated to that file.
            file_emotion.append(int(part[2]))
            file_path.append(os.path.join(Ravdess, dir, file))
        
# dataframe for emotion of files
emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])

# dataframe for path of files.
path_df = pd.DataFrame(file_path, columns=['Path'])
Ravdess_df = pd.concat([emotion_df, path_df], axis=1)

# changing integers to actual emotions.
# We map classes to be compatible with the local web interface ('fearful', 'disgusted', 'surprised')
Ravdess_df.Emotions.replace({1:'neutral', 2:'calm', 3:'happy', 4:'sad', 5:'angry', 6:'fearful', 7:'disgusted', 8:'surprised'}, inplace=True)
# Drop 'calm' (02) to maintain standard 7 emotions
Ravdess_df = Ravdess_df[Ravdess_df.Emotions != 'calm']
Ravdess_df.head()

## <center>2. Crema DataFrame</center>

In [ ]:
Crema_df = pd.DataFrame(columns=['Emotions', 'Path'])
if os.path.exists(Crema):
    crema_directory_list = os.listdir(Crema)
    file_emotion = []
    file_path = []
    for file in crema_directory_list:
        file_path.append(os.path.join(Crema, file))
        part = file.split('_')
        if len(part) >= 3:
            if part[2] == 'SAD':
                file_emotion.append('sad')
            elif part[2] == 'ANG':
                file_emotion.append('angry')
            elif part[2] == 'DIS':
                file_emotion.append('disgusted')
            elif part[2] == 'FEA':
                file_emotion.append('fearful')
            elif part[2] == 'HAP':
                file_emotion.append('happy')
            elif part[2] == 'NEU':
                file_emotion.append('neutral')
            else:
                file_emotion.append('Unknown')
    Crema_df = pd.DataFrame({'Emotions': file_emotion, 'Path': file_path})
    Crema_df = Crema_df[Crema_df.Emotions != 'Unknown']
Crema_df.head()

##  <center> 3. TESS dataset <center>

In [ ]:
Tess_df = pd.DataFrame(columns=['Emotions', 'Path'])
if os.path.exists(Tess):
    tess_directory_list = os.listdir(Tess)
    file_emotion = []
    file_path = []
    for dir in tess_directory_list:
        if not os.path.isdir(os.path.join(Tess, dir)):
            continue
        directories = os.listdir(os.path.join(Tess, dir))
        for file in directories:
            part = file.split('.')[0]
            part = part.split('_')[2]
            if part == 'ps':
                file_emotion.append('surprised')
            elif part == 'fear':
                file_emotion.append('fearful')
            elif part == 'disgust':
                file_emotion.append('disgusted')
            else:
                file_emotion.append(part)
            file_path.append(os.path.join(Tess, dir, file))
    Tess_df = pd.DataFrame({'Emotions': file_emotion, 'Path': file_path})
Tess_df.head()

##  <center> 4. SAVEE dataset <center>
The audio files in this dataset are named in such a way that the prefix letters describes the emotion classes as follows:

* 'a' = 'anger'
* 'd' = 'disgust'
* 'f' = 'fear'
* 'h' = 'happiness'
* 'n' = 'neutral'
* 'sa' = 'sadness'
* 'su' = 'surprise'

In [ ]:
Savee_df = pd.DataFrame(columns=['Emotions', 'Path'])
if os.path.exists(Savee):
    savee_directory_list = os.listdir(Savee)
    file_emotion = []
    file_path = []
    for file in savee_directory_list:
        file_path.append(os.path.join(Savee, file))
        part = file.split('_')[1]
        ele = part[:-6]
        if ele == 'a':
            file_emotion.append('angry')
        elif ele == 'd':
            file_emotion.append('disgusted')
        elif ele == 'f':
            file_emotion.append('fearful')
        elif ele == 'h':
            file_emotion.append('happy')
        elif ele == 'n':
            file_emotion.append('neutral')
        elif ele == 'sa':
            file_emotion.append('sad')
        else:
            file_emotion.append('surprised')
    Savee_df = pd.DataFrame({'Emotions': file_emotion, 'Path': file_path})
Savee_df.head()

In [ ]:
# creating Dataframe using all the dataframes we created so far.
data_path = pd.concat([Ravdess_df, Crema_df, Tess_df, Savee_df], axis = 0)
data_path.to_csv("data_path.csv", index=False)
data_path.head()

## Data Visualisation and Exploration
First let's plot the count of each emotions in our dataset.

In [ ]:
plt.title('Count of Emotions', size=16)
sns.countplot(data=data_path, x='Emotions')
plt.ylabel('Count', size=12)
plt.xlabel('Emotions', size=12)
sns.despine(top=True, right=True, left=False, bottom=False)
plt.show()

We can also plot waveplots and spectrograms for audio signals:

* **Waveplots** show the amplitude (loudness) of the audio at a given time.
* **Spectrograms** show the frequency spectrum of sound signals as they vary with time.

In [ ]:
def create_waveplot(data, sr, emotion):
    plt.figure(figsize=(10, 3))
    plt.title(f'Waveplot for audio with {emotion.upper()} emotion', size=15)
    librosa.display.waveshow(data, sr=sr)
    plt.show()

def create_spectrogram(data, sr, emotion):
    # stft function converts the data into short term fourier transform
    X = librosa.stft(data)
    Xdb = librosa.amplitude_to_db(abs(X))
    plt.figure(figsize=(12, 3))
    plt.title(f'Spectrogram for audio with {emotion.upper()} emotion', size=15)
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')   
    plt.colorbar(format='%+2.0f dB')
    plt.show()

In [ ]:
emotion = 'fearful'
sample_path = np.array(data_path.Path[data_path.Emotions == emotion])[1]
data, sampling_rate = librosa.load(sample_path, sr=22050)
create_waveplot(data, sampling_rate, emotion)
create_spectrogram(data, sampling_rate, emotion)
Audio(sample_path)

## Audio Feature Extraction

We load and extract features from the audio using our shared `features.py` module. 
This guarantees that the exact same preprocessing (trimming silence and peak normalization) is applied during training and backend API prediction.

In [ ]:
from features import preprocess, extract_features, augmentations, SR, N_FEATURES

print("Extracting features...")
X_list = []
y_list = []

for idx, row in data_path.iterrows():
    try:
        sig, sr = librosa.load(row['Path'], sr=SR)
        sig = preprocess(sig, sr)
        if sig.size < SR // 2:
            continue
        feat = extract_features(sig, sr)
        X_list.append(feat)
        y_list.append(row['Emotions'])
    except Exception as e:
        pass

X = np.array(X_list)
y = np.array(y_list)
print(f"Extracted feature shape: {X.shape}")

## Model Training Prep

In [ ]:
X_train, X_test, y_train_str, y_test_str = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_str)
y_test = label_encoder.transform(y_test_str)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Applying SMOTE to balance class distributions...")
X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f"Training size after SMOTE: {X_train.shape}")

## Build and Train CNN Model

In [ ]:
# Reshape for 1D CNN
X_train_cnn = X_train.reshape((X_train.shape[0], N_FEATURES, 1))
X_test_cnn = X_test.reshape((X_test.shape[0], N_FEATURES, 1))
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=7)

model = Sequential([
    Input(shape=(N_FEATURES, 1)),
    Conv1D(64, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Conv1D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(7, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.5, min_lr=1e-6)

print("Training CNN...")
history = model.fit(X_train_cnn, y_train_cat, validation_split=0.1, epochs=50,
                    batch_size=32, callbacks=[early_stop, reduce_lr], verbose=1)

## Model Evaluation

In [ ]:
preds = np.argmax(model.predict(X_test_cnn), axis=1)
acc = accuracy_score(y_test, preds)
print(f"CNN Test Accuracy: {acc * 100:.2f}%")
print(classification_report(y_test, preds, target_names=label_encoder.classes_))

In [ ]:
# Plot Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt='d',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, cmap='Blues')
plt.title('1D CNN Confusion Matrix')
plt.ylabel('True Emotion')
plt.xlabel('Predicted Emotion')
plt.tight_layout()
plt.show()

## Vocal Prosody Testing Instructions

To test the accuracy on your own voice, try saying the following sentences into the web recorder using the specific vocal styling:

1. **Angry** 😡: *"Kids are talking by the door!"*
   - **Vocal styling**: Shout with high volume, speed, and sharp tension.
2. **Sad** 😢: *"Dogs are sitting by the door..."*
   - **Vocal styling**: Soft, low-pitched, slow pace, with breathy sighs.
3. **Happy** 😊: *"Kids are talking by the door!"*
   - **Vocal styling**: High-pitched, enthusiastic, speak while smiling.
4. **Fearful** 😨: *"Dogs are sitting by the door?!"*
   - **Vocal styling**: High pitch, trembling/shaky voice, and short gasps.
5. **Surprised** 😲: *"Oh! Kids are talking by the door!"*
   - **Vocal styling**: Quick, wide pitch jumps, gasping tone.